In [4]:
import os
import glob
import numpy as np
import pandas as pd
import pickle

# import Python3_OpenOE_AC_map_functions_v1_08_30s as oem
import mz_LFP_functions as mz_LFP


# Load some necessary variables

In [5]:
insert_depth = 3100  #change this as appropriate

sp_bw_ch = 20/2

surface_ch = np.round(insert_depth/sp_bw_ch)
V1_hip_ch = np.round((insert_depth-1000)/sp_bw_ch)
Hip_thal_ch = np.round((insert_depth-1000-1200)/sp_bw_ch)

CA1_DG_ch = np.round((insert_depth-1000-600)/sp_bw_ch)

print(surface_ch, V1_hip_ch, Hip_thal_ch, CA1_DG_ch)

310.0 210.0 90.0 150.0


In [6]:
samples_tr = 7350 #this is based on the shortest #samples in a trial
sr = 2500
n_chan = 384
rec_length = 3.0 #how long is the arduino triggered?

---

# For multiple mice

In [9]:
data_choice = input('Scenario (1sec, 2sec, novel): ')

if data_choice == '1sec':
    start_path_ls=glob.glob(r"G:/Neuropixels/interval_operant_training/operant/raw_data/"+'oper*')     # 1 second interval training
    total_rec = 156
elif data_choice == '2sec':
    start_path_ls=glob.glob(r"G:/Neuropixels/interval_operant_training/operant/raw2_data/"+'oper*')     # 2 second interval training
    total_rec = 156
elif data_choice == 'novel':
    start_path_ls=glob.glob(r"G:/Neuropixels/interval_operant_training/operant/raw_data/"+'novel*')     # novel stimulus
    total_rec = 55
else:
    raise Exception('Input is not one of the options (1sec, 2sec, novel)')


end_path = r"\continuous\Neuropix-PXI-100.1\continuous.dat"

Scenario (1sec, 2sec, novel): 1sec


In [10]:
alldatals=[]
CC_ls = []
et_ls = []
for start_path in start_path_ls:
    ls = []
    CC_num = start_path.split('\\')[-1].split('_')[1]
    et = start_path.split('\\')[-1].split('_')[1] + "_" + start_path.split('\\')[-1].split('_')[2]
    
    for i in range(6, total_rec): # remember, you have to exclude the first short TTL signal trials
        exp_rec_path = rf"\experiment1\recording{i}"
        fileName = start_path + exp_rec_path + end_path
        data = np.memmap(fileName, dtype='int16', mode='c')
        data2 = data.reshape(-1, n_chan)
        ls.append(data2[:samples_tr, 0:384])
        
    alldatals.append(ls)
    CC_ls.append(CC_num)
    et_ls.append(et)

# alldatals #18 x 150 x 7350 x 384 or rather num_mice x num_trials x num_samples x num_ch
num_mice = len(CC_ls)
et_ls


['CC067431_HP2',
 'CC067432_HP3',
 'CC067432_HP4',
 'CC082257_HP3',
 'CC082260_HP2',
 'CC082260_HP3',
 'CC082260_HP4',
 'CC082263_HP1',
 'CC082263_HP2',
 'CC082263_HP3',
 'CC084621_HP1',
 'CC084621_HP2',
 'CC067489_HP2',
 'CC067489_HP3',
 'CC082255_HP0',
 'CC082255_HP3',
 'CC082257_HP1',
 'CC082257_HP2']

---

# Split the recordings by case

Ignore these two cells if you're just looking at the novel data

In [11]:
#Psuedo random presentation of the stimuli - 25 per row * 6 rows = 150 trials
# 0 -- drifting grating, rewarded stimulus, 100 trials
# 1 -- pink noise, unrewarded stimulus, 50 trials
stim_order = [0,1,1,0,0,1,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,
               0,1,0,1,0,1,0,0,1,1,0,0,0,1,0,1,0,0,1,1,0,0,0,0,1,
               0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,1,0,0,0,0,1,
               0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,
               1,0,0,0,1,0,1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,1,1,0,0,
               0,0,1,0,0,1,0,0,1,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0]

#Psuedo random distribution of water to the rewarded stimuli
# 0 -- water given -- 80 times
# 1 -- no water given -- 20 times
rew_order = [0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,
               0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,
               0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,
               0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0]

overall_order = []
i=0
for idx,val in enumerate(stim_order):
    if val == 0:
        i = i+1
        if rew_order[i-1] == 0:
            overall_order.append('0') # rewH20
        elif rew_order[i-1] == 1:
            overall_order.append('1') # rewnoH20
    elif val == 1:
        overall_order.append('2')     # unrew
print(len(overall_order))

150


In [12]:
rew_ls = [[] for i in range(len(alldatals))]
rew2_ls = [[] for i in range(len(alldatals))]
no_rew_ls = [[] for i in range(len(alldatals))]


for ii in range(len(alldatals)):
    for idx,val in enumerate(overall_order):
        if val == '0':
            rew_ls[ii].append(alldatals[ii][idx])
        elif val == '1':
            rew2_ls[ii].append(alldatals[ii][idx])
        elif val == '2':
            no_rew_ls[ii].append(alldatals[ii][idx])
        
        
print('rewarded trials with h2o: {0}'.format(len(rew_ls[0])))
print('rewarded trials without h2o: {0}'.format(len(rew2_ls[0])))
print('unrewarded trials: {0}'.format(len(no_rew_ls[0])))

rewarded trials with h2o: 80
rewarded trials without h2o: 20
unrewarded trials: 50


---

# Transform the lists into CAR filtered arrays
### First, define some functions

In [13]:
times = np.linspace(0, samples_tr/sr, samples_tr)
scale_factor = 0.195
time_arr = [0,0.5,1,1.5,2,2.5]
time_plot = [i*sr for i in time_arr]

In [14]:
# uses "applyCAR" and "notch_filt" functions
def create_all_arr(alldatals):
    novel_ls=[]
    for i in range(len(alldatals)):
        tmp2 = alldatals[i]  # getting the individual trial
        tmp3_ls = []
        for ii in range(len(tmp2)):
            tmp3 = tmp2[ii]
            tmp3 = tmp3.T
            filt_tmp3 = []
            for ch in range(tmp3.shape[0]):
                ch_data = tmp3[ch,:]
                ch_notc_data = mz_LFP.notch_filt(ch_data)
                filt_tmp3.append(ch_notc_data)
            filt_tmp3 = np.array(filt_tmp3)
            CARfilt_tmp3 = mz_LFP.applyCAR(filt_tmp3, pr=0)
            scaled_CARfilt_tmp3 = CARfilt_tmp3*scale_factor
            tmp3_ls.append(scaled_CARfilt_tmp3)
        novel_ls.append(tmp3_ls)
        print('Loaded {0}'.format(i))
    final_arr = np.array(novel_ls)
    return final_arr

In [11]:
# this function creates final CAR filtered arrays for rew, rew2, & unrew
# uses "applyCAR" and "notch_filt" functions
def create_rew_arr(reward_list):
    rew_ls=[]
    for ii in range(len(reward_list)):
        tmp2 = np.mean(reward_list[ii], axis = 0)           # getting the mean traces over all trials
        tmp2 = tmp2.T
        filt_tmp2 = []
        for ch in range(tmp2.shape[0]):
            ch_data = tmp2[ch,:]
            ch_notc_data = mz_LFP.notch_filt(ch_data)
            filt_tmp2.append(ch_notc_data)
        filt_tmp2 = np.array(filt_tmp2)
        CARfilt_tmp2 = mz_LFP.applyCAR(filt_tmp2, pr=0)
        scaled_CARfilt_tmp2 = CARfilt_tmp2*scale_factor
        rew_ls.append(scaled_CARfilt_tmp2)
        print('Loaded mouse {0}'.format(ii))
    rew_arr = np.array(rew_ls)
    return rew_arr

In [9]:
# this function is similar to "create_rew_arr" (^above) but for novel data
# uses "applyCAR" and "notch_filt" functions
def create_novel_arr(alldatals):
    novel_ls=[]
    for i in range(len(alldatals)):
        tmp2 = np.mean(alldatals[i], axis = 0)           # getting the mean traces over all trials
        tmp2 = tmp2.T
        filt_tmp2 = []
        for ch in range(tmp2.shape[0]):
            ch_data = tmp2[ch,:]
            ch_notc_data = mz_LFP.notch_filt(ch_data)
            filt_tmp2.append(ch_notc_data)
        filt_tmp2 = np.array(filt_tmp2)
        CARfilt_tmp2 = mz_LFP.applyCAR(filt_tmp2, pr=0)
        scaled_CARfilt_tmp2 = CARfilt_tmp2*scale_factor
        novel_ls.append(scaled_CARfilt_tmp2)
        print('Loaded {0}'.format(i))
    final_arr = np.array(novel_ls)
    return final_arr

In [15]:
len(alldatals[0][0][0]) # # mice x # trials x # samples x # channels

384

In [16]:
if (data_choice == '1sec') | (data_choice == '2sec'):
    all_rew_arr = create_all_arr(rew_ls)
    all_rew2_arr = create_all_arr(rew2_ls)
    all_norew_arr = create_all_arr(no_rew_ls)
elif data_choice == 'novel':
    all_nov_arr = create_all_arr(alldatals)

Loaded 0
Loaded 1
Loaded 2
Loaded 3
Loaded 4
Loaded 5
Loaded 6
Loaded 7
Loaded 8
Loaded 9
Loaded 10
Loaded 11
Loaded 12
Loaded 13
Loaded 14
Loaded 15
Loaded 16
Loaded 17


MemoryError: Unable to allocate 30.3 GiB for an array with shape (18, 80, 384, 7350) and data type float64

---

# Saving the LFP arrays and CC_ls
This saves each of the 3 scenarios and the cage # list for reference later

In [ ]:
all_rew_arr.shape

In [15]:


arr_save_path = r"G:\Neuropixels\interval_operant_training\operant\SAVED_DFS"



In [ ]:
if data_choice == '1sec':
    fn1 = r"1sec_alltrl_rew"
    fn2 = r"1sec_alltrl_rew2"
    fn3 = r"1sec_alltrl_unrew"
    out1 = arr_save_path + "\\" + fn1
    out2 = arr_save_path + "\\" + fn2
    out3 = arr_save_path + "\\" + fn3
    np.save(out1, all_rew_arr)         # saving rew array
    np.save(out2, all_rew2_arr)        # saving rew2 array
    np.save(out3, all_norew_arr)       # saving unrew array
    
    pkl_fn = r"1sec_cctrl_ls"
    
elif data_choice == '2sec':
    fn1 = r"2sec_alltrl_rew"
    fn2 = r"2sec_alltrl_rew2"
    fn3 = r"2sec_alltrl_unrew"
    out1 = arr_save_path + "\\" + fn1
    out2 = arr_save_path + "\\" + fn2
    out3 = arr_save_path + "\\" + fn3
    np.save(out1, all_rew_arr)         # saving rew array
    np.save(out2, all_rew2_arr)        # saving rew2 array
    np.save(out3, all_norew_arr)       # saving unrew array
    
    pkl_fn = r"2sec_cctrl_ls"
    
elif data_choice == 'novel':
    fn1 = r"novel_alltrl_rew"
    out1 = arr_save_path + "\\" + fn1
    np.save(out1, all_nov_arr)         # saving novel array
    
    pkl_fn = r"novel_cctrl_ls"

In [ ]:
pkl_out = arr_save_path + "\\" + pkl_fn

open_file = open(pkl_out, "wb")
pickle.dump(CC_ls, open_file)
open_file.close()

---

---

---